In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

DEBUG - URL: https://test.dbrepo.tuwien.ac.at//api/v1/container
DEBUG - Method: get
DEBUG - Headers: {}
DEBUG - Payload type: <class 'NoneType'>
[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [2]:
container_id = '6cfb3b8e-1792-4e46-871a-f3d103527203'
DB_ID = "cf27a11d-58e5-4693-856c-e8f3527e3394"

In [3]:
from dbrepo.api.dto import (
    CreateIdentifier,
    CreateIdentifierTitle,
    CreateIdentifierDescription,
    RelatedIdentifier,
    RelatedIdentifierType,
    RelatedIdentifierRelation,
    CreateIdentifierCreator,
    CreateRelatedIdentifier,
    DescriptionType,
    IdentifierType,
    License,
    Language
)

def build_consolidated_use_case_pid(database_id) -> CreateIdentifier:
    """
    Generates DBRepo metadata
    """
    
    titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
            language=Language.EN
        )
    ]
    
    # 2. Comprehensive Metadata Descriptions (Abstract, Methods, Scope, Units)
    descriptions = [
        CreateIdentifierDescription(
            description=(
                """Abstract: This use case explores the correlation relationship between 
                illicit drug use in major European cities and their regional economic productivity (GDP per capita). 
                Original Publishers: EUDA & SCORE, EUROSTAT.
                EUDA & SCORE: URI: https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en
                EUROSTAT: DOI: https://doi.org/10.2908/NAMA_10R_3GDP, URI: https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_20659344/default/table"""
            ),
            type=DescriptionType("Abstract"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Data Stewardship and Preprocessing Challenge: While the drug dataset identifies locations 
                by specific city strings (e.g., 'Graz', 'Steyr'), Eurostat utilizes standardized NUTS-3 administrative codes 
                (e.g., 'DE212'). This is resolved via a custom mapping schema table ('city_map'.
                Only the active filtered subset utilized in this longitudinal frame is republished here."""
            ),
            type="Methods",
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Temporal & Spatial Coverage: 
                Wastewater tracking spans annually from 2011 to 2025 across 115 cities and 25 countries in the European Union, 
                Norway, and Türkiye. GDP tracking spans annually from 2000 to 2024 across EU Member States, Candidate and 
                potential Candidate Countries, Norway, and Switzerland."""
            ),
            type=DescriptionType("TechnicalInfo"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Units of Measure: "
                Wastewater metrics indicate concentrations (mg/1000p/day) of illicit drug loads (Cocaine, Methamphetamine, MDMA) 
                measured from 24-hour composite samples collected over a single week between March and May. 
                GDP values indicate economic output expressed in National Currency, Euros, Purchasing Power Standards (PPS), 
                thousands of persons/hours worked, growth rates, or Index 2020=100."""
            ),
            type=DescriptionType("Other"),
            language=Language.EN
        )
    ]
    
    # 3. Explicit Provenance & Lineage Relationships
    related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type=RelatedIdentifierType.DOI,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type=RelatedIdentifierType.URL,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        )
    ]

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
    
    # 4. Master DataCite Payload Assembly
    identifier_payload = CreateIdentifier(
        database_id=database_id,
        publication_year=2026,           # Project release date
        publisher="EUDA & SCORE, EUROSTAT",
        type = IdentifierType.DATABASE,
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for both sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )
    
    return identifier_payload

In [4]:
identifier_payload = build_consolidated_use_case_pid(database_id=DB_ID)

In [5]:
response = client._wrapper(
    method="post", 
    url=f'/api/v1/identifier',#database/{DB_ID}', 
    payload=identifier_payload
)

response.raise_for_status()

DEBUG - URL: https://test.dbrepo.tuwien.ac.at//api/v1/identifier
DEBUG - Method: post
DEBUG - Headers: {}
DEBUG - Payload type: <class 'dict'>
DEBUG - Payload keys: dict_keys(['database_id', 'type', 'creators', 'publication_year', 'publisher', 'titles', 'descriptions', 'language', 'licenses', 'related_identifiers'])
DEBUG - Payload: {'database_id': 'cf27a11d-58e5-4693-856c-e8f3527e3394', 'type': 'database', 'creators': [{'creator_name': 'Helene Vaught', 'firstname': 'Helene', 'lastname': 'Vaught', 'affiliation': 'TU Wien'}, {'creator_name': 'Vlada Hlushchenko', 'firstname': 'Vlada', 'lastname': 'Hlushchenko', 'affiliation': 'TU Wien'}, {'creator_name': 'Barnabás Paksi', 'firstname': 'Barnabás', 'lastname': 'Paksi', 'affiliation': 'TU Wien'}, {'creator_name': 'Amélie Assmayr', 'firstname': 'Amélie', 'lastname': 'Assmayr', 'affiliation': 'TU Wien'}], 'publication_year': 2026, 'publisher': 'EUDA & SCORE, EUROSTAT', 'titles': [{'title': 'Predictive Modeling of Regional GDP per Capita based

HTTPError: 400 Client Error:  for url: https://test.dbrepo.tuwien.ac.at//api/v1/identifier

In [6]:
our_titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
            language=Language.EN
        )
    ]

we_as_creators = [
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]

our_descriptions = [
        CreateIdentifierDescription(
            description=(
                """Abstract: This use case explores the correlation relationship between 
                illicit drug use in major European cities and their regional economic productivity (GDP per capita). 
                Original Publishers: EUDA & SCORE, EUROSTAT.
                EUDA & SCORE: URI: https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en
                EUROSTAT: DOI: https://doi.org/10.2908/NAMA_10R_3GDP, URI: https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_20659344/default/table"""
            ),
            type=DescriptionType.ABSTRACT,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Data Stewardship and Preprocessing Challenge: While the drug dataset identifies locations 
                by specific city strings (e.g., 'Graz', 'Steyr'), Eurostat utilizes standardized NUTS-3 administrative codes 
                (e.g., 'DE212'). This is resolved via a custom mapping schema table ('city_map'.
                Only the active filtered subset utilized in this longitudinal frame is republished here."""
            ),
            type=DescriptionType.METHODS,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Temporal & Spatial Coverage: 
                Wastewater tracking spans annually from 2011 to 2025 across 115 cities and 25 countries in the European Union, 
                Norway, and Türkiye. GDP tracking spans annually from 2000 to 2024 across EU Member States, Candidate and 
                potential Candidate Countries, Norway, and Switzerland."""
            ),
            type=DescriptionType.TECHNICAL_INFO,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Units of Measure: "
                Wastewater metrics indicate concentrations (mg/1000p/day) of illicit drug loads (Cocaine, Methamphetamine, MDMA) 
                measured from 24-hour composite samples collected over a single week between March and May. 
                GDP values indicate economic output expressed in National Currency, Euros, Purchasing Power Standards (PPS), 
                thousands of persons/hours worked, growth rates, or Index 2020=100."""
            ),
            type=DescriptionType.OTHER,
            language=Language.EN
        )
    ]
cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            """Creative Commons Attribution 4.0 International: Allows users to copy, 
            distribute, display, perform, and modify the work, even for commercial purposes, 
            provided that they give appropriate credit to the original creator."""
        )
    )

related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            #id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type=RelatedIdentifierType.DOI,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            #id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type=RelatedIdentifierType.URL,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        )
    ]

identifier_creation_resp = client.create_identifier(database_id=DB_ID,
                        type=IdentifierType.TABLE,
                        titles = our_titles,
                        publisher="EUDA & SCORE, EUROSTAT",
                        creators=we_as_creators,
                        publication_year=2026,
                        descriptions=our_descriptions,
                        licenses=[cc_by_4_0],
                        language="en",                        
                        subset_id = None, 
                        view_id = None, 
                        table_id = None,
                        related_identifiers=related_identifiers) 

DEBUG - URL: https://test.dbrepo.tuwien.ac.at//api/v1/identifier
DEBUG - Method: post
DEBUG - Headers: {}
DEBUG - Payload type: <class 'dict'>
DEBUG - Payload keys: dict_keys(['database_id', 'type', 'creators', 'publication_year', 'publisher', 'titles', 'descriptions', 'language', 'licenses', 'related_identifiers'])
DEBUG - Payload: {'database_id': 'cf27a11d-58e5-4693-856c-e8f3527e3394', 'type': 'table', 'creators': [{'creator_name': 'Helene Vaught', 'firstname': 'Helene', 'lastname': 'Vaught', 'affiliation': 'TU Wien'}, {'creator_name': 'Vlada Hlushchenko', 'firstname': 'Vlada', 'lastname': 'Hlushchenko', 'affiliation': 'TU Wien'}, {'creator_name': 'Barnabás Paksi', 'firstname': 'Barnabás', 'lastname': 'Paksi', 'affiliation': 'TU Wien'}, {'creator_name': 'Amélie Assmayr', 'firstname': 'Amélie', 'lastname': 'Assmayr', 'affiliation': 'TU Wien'}], 'publication_year': 2026, 'publisher': 'EUDA & SCORE, EUROSTAT', 'titles': [{'title': 'Predictive Modeling of Regional GDP per Capita based on

MalformedError: Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}

In [7]:
our_titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
            language=Language.EN
        )
    ]

we_as_creators = [
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]

our_descriptions = [
        CreateIdentifierDescription(
            description=(
                """Abstract: This use case explores the correlation relationship between 
                illicit drug use in major European cities and their regional economic productivity (GDP per capita). 
                Original Publishers: EUDA & SCORE, EUROSTAT.
                EUDA & SCORE: URI: https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en
                EUROSTAT: DOI: https://doi.org/10.2908/NAMA_10R_3GDP, URI: https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_20659344/default/table"""
            ),
            type=DescriptionType.ABSTRACT,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Data Stewardship and Preprocessing Challenge: While the drug dataset identifies locations 
                by specific city strings (e.g., 'Graz', 'Steyr'), Eurostat utilizes standardized NUTS-3 administrative codes 
                (e.g., 'DE212'). This is resolved via a custom mapping schema table ('city_map'.
                Only the active filtered subset utilized in this longitudinal frame is republished here."""
            ),
            type=DescriptionType.METHODS,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Temporal & Spatial Coverage: 
                Wastewater tracking spans annually from 2011 to 2025 across 115 cities and 25 countries in the European Union, 
                Norway, and Türkiye. GDP tracking spans annually from 2000 to 2024 across EU Member States, Candidate and 
                potential Candidate Countries, Norway, and Switzerland."""
            ),
            type=DescriptionType.TECHNICAL_INFO,
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Units of Measure: "
                Wastewater metrics indicate concentrations (mg/1000p/day) of illicit drug loads (Cocaine, Methamphetamine, MDMA) 
                measured from 24-hour composite samples collected over a single week between March and May. 
                GDP values indicate economic output expressed in National Currency, Euros, Purchasing Power Standards (PPS), 
                thousands of persons/hours worked, growth rates, or Index 2020=100."""
            ),
            type=DescriptionType.OTHER,
            language=Language.EN
        )
    ]
cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            """Creative Commons Attribution 4.0 International: Allows users to copy, 
            distribute, display, perform, and modify the work, even for commercial purposes, 
            provided that they give appropriate credit to the original creator."""
        )
    )

related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            #id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type=RelatedIdentifierType.DOI,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            #id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type=RelatedIdentifierType.URL,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        )
    ]

identifier_creation_resp = client.create_identifier(database_id=DB_ID,
                        type=IdentifierType.TABLE,
                        titles = our_titles,
                        publisher="EUDA & SCORE, EUROSTAT",
                        creators=we_as_creators,
                        publication_year=2026,
                        descriptions=our_descriptions,
                        licenses=[cc_by_4_0],
                        table_id ='aa6cbf8c-f35e-4411-81ae-20f1a1acf682',
                        language=Language.EN,                        
                        subset_id = None, 
                        view_id = None, 
                        related_identifiers=related_identifiers) 

DEBUG - URL: https://test.dbrepo.tuwien.ac.at//api/v1/identifier
DEBUG - Method: post
DEBUG - Headers: {}
DEBUG - Payload type: <class 'dict'>
DEBUG - Payload keys: dict_keys(['database_id', 'type', 'creators', 'publication_year', 'publisher', 'titles', 'descriptions', 'language', 'licenses', 'table_id', 'related_identifiers'])
DEBUG - Payload: {'database_id': 'cf27a11d-58e5-4693-856c-e8f3527e3394', 'type': 'table', 'creators': [{'creator_name': 'Helene Vaught', 'firstname': 'Helene', 'lastname': 'Vaught', 'affiliation': 'TU Wien'}, {'creator_name': 'Vlada Hlushchenko', 'firstname': 'Vlada', 'lastname': 'Hlushchenko', 'affiliation': 'TU Wien'}, {'creator_name': 'Barnabás Paksi', 'firstname': 'Barnabás', 'lastname': 'Paksi', 'affiliation': 'TU Wien'}, {'creator_name': 'Amélie Assmayr', 'firstname': 'Amélie', 'lastname': 'Assmayr', 'affiliation': 'TU Wien'}], 'publication_year': 2026, 'publisher': 'EUDA & SCORE, EUROSTAT', 'titles': [{'title': 'Predictive Modeling of Regional GDP per Cap

MalformedError: Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}

## NOTHING WORKS, REPRODUCING ERROR MINIMALLY

In [8]:
our_titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
            language="en"
        )
    ]

we_as_creators = [
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien")            
        ]

our_descriptions = [
        CreateIdentifierDescription(
            description="Units of Measure",
            type="Other"#,
            #language=Language.EN
        )
    ]
cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            """Creative Commons Attribution 4.0 International: Allows users to copy, 
            distribute, display, perform, and modify the work, even for commercial purposes, 
            provided that they give appropriate credit to the original creator."""
        )
    )

related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            #id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type="DOI",
            relation="IsDerivedFrom"
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            #id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type="URL",
            relation="IsDerivedFrom"
        )
    ]

identifier_creation_resp = client.create_identifier(database_id=DB_ID,
                        type="database", 
                        titles = our_titles,
                        publisher="EUDA & SCORE, EUROSTAT",
                        creators=we_as_creators,
                        publication_year=2026,
                        descriptions=our_descriptions,
                        licenses=[cc_by_4_0]
                                      
                        ) 

DEBUG - URL: https://test.dbrepo.tuwien.ac.at//api/v1/identifier
DEBUG - Method: post
DEBUG - Headers: {}
DEBUG - Payload type: <class 'dict'>
DEBUG - Payload keys: dict_keys(['database_id', 'type', 'creators', 'publication_year', 'publisher', 'titles', 'descriptions', 'licenses'])
DEBUG - Payload: {'database_id': 'cf27a11d-58e5-4693-856c-e8f3527e3394', 'type': 'database', 'creators': [{'creator_name': 'Helene Vaught', 'firstname': 'Helene', 'lastname': 'Vaught', 'affiliation': 'TU Wien'}], 'publication_year': 2026, 'publisher': 'EUDA & SCORE, EUROSTAT', 'titles': [{'title': 'Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology', 'language': 'en'}], 'descriptions': [{'description': 'Units of Measure', 'type': 'Other'}], 'licenses': [{'identifier': 'CC-BY-4.0', 'uri': 'https://creativecommons.org/licenses/by/4.0/', 'description': 'Creative Commons Attribution 4.0 International: Allows users to copy, \n            distribute, display, perform, and modify 

MalformedError: Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}

In [19]:
from dbrepo.api.dto import CreateIdentifier

payload = CreateIdentifier(
    database_id=DB_ID,
    type="database",
    titles=our_titles,
    publisher="EUDA & SCORE, EUROSTAT",
    creators=we_as_creators,
    publication_year=2026,
    descriptions=our_descriptions,
    licenses=[cc_by_4_0]
)

print("Final payload to be sent:")
import json
print(json.dumps(payload.model_dump(), indent=2))

Final payload to be sent:
{
  "database_id": "cf27a11d-58e5-4693-856c-e8f3527e3394",
  "type": "database",
  "creators": [
    {
      "creator_name": "Helene Vaught",
      "firstname": "Helene",
      "lastname": "Vaught",
      "affiliation": "TU Wien",
      "name_type": null,
      "name_identifier": null,
      "affiliation_identifier": null
    }
  ],
  "publication_year": 2026,
  "publisher": "EUDA & SCORE, EUROSTAT",
  "titles": [
    {
      "title": "Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
      "language": "en",
      "type": null
    }
  ],
  "descriptions": [
    {
      "description": "Units of Measure",
      "language": null,
      "type": "Other"
    }
  ],
  "funders": null,
  "doi": null,
  "language": null,
  "licenses": [
    {
      "identifier": "CC-BY-4.0",
      "uri": "https://creativecommons.org/licenses/by/4.0/",
      "description": "Creative Commons Attribution 4.0 International: Allows users to copy, \n    

In [20]:
client.get_database(DB_ID)

Database(id='cf27a11d-58e5-4693-856c-e8f3527e3394', name='dast_g20_wastewater_epidemiology', exchange_name='dbrepo', internal_name='dast_g20_wastewater_epidemiology_1kph', is_public=True, is_schema_public=True, is_dashboard_enabled=False, container=ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None), owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None), contact=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None), identifiers=[], subsets=[], tables=[Table(id='aa6cbf8c-f35e-4411-81ae-20f1a1acf682', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', name='wastewater_data', owner=UserBrief(username='data_stewardship_group

In [28]:
import requests
from dbrepo.api.dto import Identifier

# Try absolute minimum - just required fields with no optional fields at all
minimal_payload = {
    "database_id": DB_ID,
    "type": "database",
    "titles": [
        {
            "title": "Test"
        }
    ],
    "publisher": "Test",
    "creators": [
        {
            "creator_name": "Test Creator"
        }
    ],
    "publication_year": 2026
}

url = f'{client.endpoint}/api/v1/identifier'
auth = (client.username, client.password) if client.username and client.password else None
headers = {}
if not auth and client.password:
    headers["Authorization"] = f"Bearer {client.password}"

response = requests.post(
    url=url, 
    auth=auth, 
    verify=client.secure,
    json=minimal_payload,
    headers=headers
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text}")

Status: 400
Response: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}


In [23]:
try:
    identifiers = client.get_identifiers(database_id=DB_ID)
    print(f"Found {len(identifiers)} identifiers")
except Exception as e:
    print(f"Error getting identifiers: {e}")

Found 0 identifiers


In [24]:
# Test authentication with a simple GET request
url = f'{client.endpoint}/api/v1/identifier'
auth = (client.username, client.password) if client.username and client.password else None
headers = {}
if not auth and client.password:
    headers["Authorization"] = f"Bearer {client.password}"

response = requests.get(
    url=url, 
    auth=auth, 
    verify=client.secure,
    headers=headers
)
print(f"GET Status: {response.status_code}")

GET Status: 200


In [27]:
import requests
import json

# Create the minimal payload manually without None values
minimal_payload = {
    "database_id": DB_ID,
    "type": "database",
    "titles": [
        {
            "title": "Test"
        }
    ],
    "publisher": "Test",
    "creators": [
        {
            "creator_name": "Test Creator"
        }
    ],
    "publication_year": 2026
}

url = f'{client.endpoint}/api/v1/identifier'
auth = (client.username, client.password) if client.username and client.password else None
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}
if not auth and client.password:
    headers["Authorization"] = f"Bearer {client.password}"

# Convert to JSON string manually to ensure no nulls
json_data = json.dumps(minimal_payload)

response = requests.post(
    url=url, 
    auth=auth, 
    verify=client.secure,
    data=json_data,  # Use data instead of json parameter
    headers=headers
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text}")

Status: 400
Response: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}
